[![img/pythonista.png](img/pythonista.png)](https://www.pythonista.io)

© Pythonista · Licencia CC BY 4.0 — https://creativecommons.org/licenses/by/4.0/deed.es

&copy; José Luis Chiquete Valdivieso. 2017-2026.

# Temperatura y modo de razonamiento.

**py521 · IA Generativa en Entornos Locales** · Pythonista® · Instructor: José Luis Chiquete · Demo del Tema 2: *Cómo genera texto un modelo*

En este *notebook* se enviará la misma petición con temperatura baja y alta, y después con el modo de razonamiento desactivado y activado. Se comparará cuánto varían las respuestas y cuánto tarda en aparecer cada una.

## 🧭 Cómo usar este *notebook*.

* Ejecuta las celdas **en orden** (`Shift + Enter`).
* El modelo y el servicio se eligen con variables de entorno (`LLM_MODEL`, `LLM_BASE_URL`); por omisión, `qwen3:1.7b` en *Ollama*.
* Necesitas un modelo con modo de razonamiento para la sección 2 (por ejemplo, `qwen3:1.7b` o `gemma4:e2b`).
* Los tiempos dependen del equipo: lo importante es la comparación entre casos, no el valor absoluto.

### Configuración común.

Esta celda es **idéntica en todos los *notebooks***: lee la configuración de variables de entorno (o del archivo `.env`, si existe) y usa valores por omisión cuando no están definidas. Muestra todas las variables, pero **en esta demo importan: `LLM_BASE_URL` y `LLM_MODEL`**. Las claves solo se muestran como «definida / no definida». La tabla completa de variables está en el `README.md` y en `00_preliminares_entorno_y_secretos.ipynb`.

In [ ]:
# Configuración común de py521 (celda idéntica en todos los notebooks).
import os
from pathlib import Path

from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv(usecwd=True), override=False)  # las variables del sistema tienen prioridad sobre .env

def _entorno(nombre, por_omision=None):
    valor = os.environ.get(nombre, "").strip()
    return valor or por_omision

# Servicio compatible con OpenAI (Ollama, LM Studio, llama.cpp server, vLLM...).
LLM_BASE_URL = _entorno("LLM_BASE_URL", _entorno("LLM_DEFAULT_BASE_URL", "http://localhost:11434/v1")).rstrip("/")  # el contenedor fija LLM_DEFAULT_BASE_URL
LLM_API_KEY = _entorno("LLM_API_KEY")              # secreto: sin valor por omisión
LLM_MODEL = _entorno("LLM_MODEL", "qwen3:1.7b")
LLM_EMBED_MODEL = _entorno("LLM_EMBED_MODEL", "qwen3-embedding:0.6b")
LLM_MODEL_A = _entorno("LLM_MODEL_A", "qwen3:1.7b-q4_K_M")
LLM_MODEL_B = _entorno("LLM_MODEL_B", "qwen3:1.7b-q8_0")
LLM_CONTEXT_A = int(_entorno("LLM_CONTEXT_A", "4096"))
LLM_CONTEXT_B = int(_entorno("LLM_CONTEXT_B", "16384"))
RAG_DOCS_DIR = Path(_entorno("RAG_DOCS_DIR", "data"))
LLM_PROXY_KEY = _entorno("LLM_PROXY_KEY")          # secreto de la demo 05: sin valor por omisión
LLM_PROXY_HOST = _entorno("LLM_PROXY_HOST", "127.0.0.1")
LLM_PROXY_PORT = int(_entorno("LLM_PROXY_PORT", "8800"))

# API nativa de Ollama (solo para las funciones marcadas «específico de Ollama»).
# Se usa OLLAMA_HOST, como el cliente oficial de Ollama; si no existe, se deriva de LLM_BASE_URL.
def _url_ollama(valor):
    valor = (valor or LLM_BASE_URL.removesuffix("/v1")).strip().rstrip("/")
    if "://" not in valor:
        anfitrion = valor.split("/")[0]
        valor = "http://" + (valor if ":" in anfitrion.split("]")[-1] else valor.replace(anfitrion, anfitrion + ":11434", 1))
    return valor.replace("://0.0.0.0", "://localhost")  # 0.0.0.0 es una dirección de escucha, no de destino

OLLAMA_URL = _url_ollama(_entorno("OLLAMA_HOST"))

for nombre in ["LLM_BASE_URL", "LLM_MODEL", "LLM_EMBED_MODEL", "LLM_MODEL_A", "LLM_MODEL_B",
               "LLM_CONTEXT_A", "LLM_CONTEXT_B", "RAG_DOCS_DIR", "LLM_PROXY_HOST", "LLM_PROXY_PORT", "OLLAMA_URL"]:
    print(f"{nombre:<16} {globals()[nombre]}")
for nombre in ["LLM_API_KEY", "LLM_PROXY_KEY", "HF_TOKEN"]:
    print(f"{nombre:<16} {'definida' if _entorno(nombre) else 'no definida'}")

### Preparación del entorno.

* La siguiente celda creará el cliente de la API compatible con *OpenAI*, comprobará que el modelo está disponible y definirá la función `preguntar()`, que mide el tiempo al primer token y los tokens por segundo.

In [ ]:
from openai import OpenAI

cliente = OpenAI(base_url=LLM_BASE_URL, api_key=LLM_API_KEY or "sin-clave")  # Ollama y LM Studio sin autenticación ignoran la clave

def comprobar_servicio(modelos):
    """Comprueba que el servicio responde (GET /models) y que ofrece los modelos indicados."""
    try:
        disponibles = {m.id for m in cliente.models.list().data}
    except Exception as error:
        raise RuntimeError(f"No responde el servicio en {LLM_BASE_URL} ({type(error).__name__}). "
                           "Abre Ollama o el motor que uses, o revisa LLM_BASE_URL.") from None
    faltan = [m for m in modelos if m not in disponibles and f"{m}:latest" not in disponibles]
    if faltan:
        raise RuntimeError(f"El servicio no ofrece: {', '.join(faltan)}. En Ollama: "
                           + " · ".join(f"ollama pull {m}" for m in faltan))
    print(f"Servicio en {LLM_BASE_URL}: responde. Modelos: {', '.join(modelos)}.")

comprobar_servicio([LLM_MODEL])

In [ ]:
import time

def preguntar(prompt, modelo=None, temperatura=None, razonamiento=None, max_tokens=None, sistema=None):
    """Envía una petición a /chat/completions en modo streaming y mide los tiempos.

    razonamiento: None (valor del modelo), "none" (desactivado) o "low" / "medium" / "high".
    Se envía como `reasoning_effort` (Ollama, vLLM, LM Studio) y como
    `chat_template_kwargs.enable_thinking` (llama.cpp server, vLLM con Qwen3); cada motor ignora lo que no usa.
    """
    mensajes = ([{"role": "system", "content": sistema}] if sistema else []) + [{"role": "user", "content": prompt}]
    extra, argumentos = {}, {}
    if temperatura is not None:
        argumentos["temperature"] = temperatura
    if max_tokens is not None:
        argumentos["max_tokens"] = max_tokens
    if razonamiento is not None:
        argumentos["reasoning_effort"] = razonamiento
        extra["chat_template_kwargs"] = {"enable_thinking": razonamiento != "none"}
    inicio = time.perf_counter()
    primer_token = primer_texto = None
    texto, pensado, uso, fin = [], [], None, None
    flujo = cliente.chat.completions.create(model=modelo or LLM_MODEL, messages=mensajes, stream=True,
                                            stream_options={"include_usage": True}, extra_body=extra or None,
                                            **argumentos)
    for parte in flujo:
        if parte.usage:
            uso = parte.usage
        if not parte.choices:
            continue
        delta = parte.choices[0].delta
        extras = delta.model_extra or {}
        trozo_razonamiento = extras.get("reasoning") or extras.get("reasoning_content")  # el nombre varía por motor
        ahora = time.perf_counter() - inicio
        if primer_token is None and (delta.content or trozo_razonamiento):
            primer_token = ahora
        if primer_texto is None and delta.content:
            primer_texto = ahora
        pensado.append(trozo_razonamiento or "")
        texto.append(delta.content or "")
        fin = parte.choices[0].finish_reason or fin
    total = time.perf_counter() - inicio
    tokens = uso.completion_tokens if uso else 0
    generacion = total - (primer_token or total)
    return {
        "respuesta": "".join(texto).strip(),
        "razonamiento": "".join(pensado).strip(),
        "primer_token_s": primer_token,
        "primer_texto_s": primer_texto,
        "total_s": total,
        "tokens": tokens,
        "tokens_s": tokens / generacion if generacion > 0 else 0.0,
        "fin": fin,
    }

* La siguiente celda fija los valores de la demo: 5 peticiones por caso, temperatura baja 0 y temperatura alta 1,2.

In [ ]:
import matplotlib.pyplot as plt

REPETICIONES = 5        # peticiones por caso
TEMPERATURA_BAJA = 0.0  # en muchas herramientas, 0 equivale a elegir siempre el token más probable
TEMPERATURA_ALTA = 1.2

## 1. Temperatura baja frente a temperatura alta.

* La temperatura cambia la diferencia entre las probabilidades de los candidatos a siguiente token.
* **Baja:** casi siempre se elige el token más probable; las respuestas se repiten.
* **Alta:** los candidatos menos probables se eligen más a menudo; las respuestas varían.
* En esta sección se desactiva el razonamiento para que solo cambie la temperatura.

In [ ]:
PROMPT_TEMPERATURA = "Propón un título de máximo ocho palabras para un artículo sobre ahorro de energía en casa. Responde solo con el título."

resultados_temperatura = {}
for nombre, temperatura in [("baja", TEMPERATURA_BAJA), ("alta", TEMPERATURA_ALTA)]:
    resultados_temperatura[nombre] = [
        preguntar(PROMPT_TEMPERATURA, temperatura=temperatura, razonamiento="none", max_tokens=40)
        for _ in range(REPETICIONES)
    ]
    print(f"\nTemperatura {nombre} ({temperatura}):")
    for i, r in enumerate(resultados_temperatura[nombre], 1):
        print(f"  {i}. {r['respuesta']}")

### 1.1 Cuánto varían las respuestas.

* Se cuenta cuántas respuestas **distintas** hubo en cada caso y el tiempo medio de respuesta.

In [ ]:
distintas = {n: len({r["respuesta"] for r in rs}) for n, rs in resultados_temperatura.items()}
tiempo_medio = {n: sum(r["total_s"] for r in rs) / len(rs) for n, rs in resultados_temperatura.items()}

fig, ejes = plt.subplots(1, 2, figsize=(9, 3.5))
ejes[0].bar(list(distintas), list(distintas.values()), color=["#4C72B0", "#DD8452"])
ejes[0].set_title(f"Respuestas distintas de {REPETICIONES}")
ejes[0].set_ylim(0, REPETICIONES)
ejes[1].bar(list(tiempo_medio), list(tiempo_medio.values()), color=["#4C72B0", "#DD8452"])
ejes[1].set_title("Tiempo medio por respuesta (s)")
for eje in ejes:
    eje.set_xlabel("Temperatura")
plt.tight_layout()
plt.show()

* La temperatura cambia la **variedad**, no la exactitud ni la velocidad: una temperatura baja da respuestas más repetibles, no más verdaderas.
* Para extraer datos, clasificar o resumir conviene una temperatura baja, salvo que la ficha del modelo indique otra cosa.

## 2. Modo de razonamiento desactivado frente a activado.

* Con el modo de razonamiento activo, el modelo genera tokens de pensamiento **antes** de la respuesta visible.
* Esos tokens cuentan dentro de la longitud máxima de la respuesta y de la ventana de contexto, y alargan la espera.
* Se repite la misma pregunta con razonamiento `"none"` (desactivado) y `"high"` (activado).
* Sin razonamiento se usa temperatura 0. Con razonamiento se usa la temperatura que recomienda la ficha del modelo (si no se envía, el motor usa la de la ficha; en *Ollama* es 0,6): la ficha de *Qwen3* desaconseja la temperatura 0 en ese modo porque puede degradar la respuesta o repetirse sin fin.
* Cada intento cambia la primera línea de la petición para que cada medición empiece desde cero (el motor guarda lo que ya procesó y respondería al instante).

In [ ]:
PREGUNTA = "¿Cuánto suman los múltiplos de 3 entre 1 y 30, incluidos? Responde solo con el número."

RESPUESTA_ESPERADA = "165"  # 3 + 6 + 9 + ... + 30

casos = {}
for i, (nombre, nivel, temperatura) in enumerate([("desactivado", "none", 0.0), ("activado", "high", None)], 1):
    casos[nombre] = preguntar(f"Consulta {i}.\n{PREGUNTA}", temperatura=temperatura, razonamiento=nivel, max_tokens=2048)
    r = casos[nombre]
    acierto = "✅" if RESPUESTA_ESPERADA in r["respuesta"] else "❌"
    print(f"Razonamiento {nombre}: {r['respuesta']} {acierto}")
    print(f"  caracteres de razonamiento: {len(r['razonamiento'])} · tokens generados: {r['tokens']} · "
          f"primer texto visible: {r['primer_texto_s'] or 0:.1f} s · total: {r['total_s']:.1f} s\n")

* ✅ / ❌ compara con la respuesta correcta (165). Un modelo pequeño puede equivocarse con razonamiento o sin él: el razonamiento suele ayudar en tareas de varios pasos, pero no garantiza la respuesta. Lo que siempre cambia es el tiempo y los tokens.
* Sin razonamiento (temperatura 0) la respuesta se repite en cada ejecución. Con razonamiento la temperatura no es 0 y la respuesta puede variar: en las pruebas de preparación con `qwen3:1.7b` acertó en la mayoría de los intentos, no en todos. Si sale ❌, vuelve a ejecutar la celda.
* El razonamiento puede aparecer en otro idioma que la respuesta; depende del modelo.
* Si el razonamiento activado no muestra caracteres de razonamiento, el modelo o el motor no lo separan en un campo propio, o el modelo no tiene ese modo.
* La siguiente celda mostrará el inicio del razonamiento.

In [ ]:
print(casos["activado"]["razonamiento"][:800] or "(sin razonamiento separado)")

### 2.1 Comparación de tiempos y tokens.

In [ ]:
nombres = list(casos)
fig, ejes = plt.subplots(1, 3, figsize=(11, 3.5))
datos = [("Tokens generados", "tokens"), ("Primer texto visible (s)", "primer_texto_s"), ("Tiempo total (s)", "total_s")]
for eje, (titulo, clave) in zip(ejes, datos):
    eje.bar(nombres, [casos[n][clave] or 0 for n in nombres], color=["#4C72B0", "#C44E52"])
    eje.set_title(titulo)
    eje.set_xlabel("Razonamiento")
plt.tight_layout()
plt.show()

### 2.2 El razonamiento ocupa la longitud máxima.

* Con una longitud máxima corta (`max_tokens`), el razonamiento puede consumirla entera y la respuesta visible queda vacía o cortada.
* `fin` indica por qué terminó la generación: `stop` (terminó sola) o `length` (llegó al límite).

In [ ]:
corto = preguntar(f"Consulta 3.\n{PREGUNTA}", razonamiento="high", max_tokens=64)
print(f"Respuesta visible: {corto['respuesta'] or '(vacía)'}")
print(f"Tokens generados: {corto['tokens']} · fin: {corto['fin']}")

* Práctica: subir la longitud máxima cuando el razonamiento esté activo, o bajar el esfuerzo si la respuesta se corta.

---
## ✅ Cierre.

* La temperatura cambia cuánto varían las respuestas; no las hace más exactas.
* El modo de razonamiento puede mejorar las respuestas difíciles, pero genera más tokens, ocupa longitud y contexto, y retrasa la respuesta visible.

**Siguiente *notebook*:** `02_ejecutar_y_comprobar_un_modelo_local.ipynb`.

<p style="text-align: center"><a rel="license" href="https://creativecommons.org/licenses/by/4.0/deed.es"><img alt="Licencia Creative Commons" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/80x15.png" /></a><br />Esta obra está bajo una <a rel="license" href="https://creativecommons.org/licenses/by/4.0/deed.es">Licencia Creative Commons Atribución 4.0 Internacional</a>.</p>
<p style="text-align: center">&copy; Pythonista · Licencia CC BY 4.0 — https://creativecommons.org/licenses/by/4.0/deed.es</p>
<p style="text-align: center">&copy; José Luis Chiquete Valdivieso. 2017-2026.</p>